# Evaluating and Observing Agents: Outcomes, Trajectories, Traces and Spans

The full code of [Evaluating and Observing Agents: Outcomes, Trajectories, Traces and Spans](https://residual-stream.lvtan71.workers.dev/posts/evaluating-and-observing-agents/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/evaluating-and-observing-agents) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/evaluating-and-observing-agents.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 6 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
%pip install -q "numpy==2.*" "torch==2.14.*" "transformers==5.17.*" huggingface_hub

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "evaluating-and-observing-agents"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

## `pass_k.py`

pass@k and pass^k for one task at several per-attempt success rates, and their unbiased estimates from n attempts with c successes.

In [ ]:
!{sys.executable} pass_k.py

The post printed:

```text
    p   pass@1   pass@4   pass@8     pass^1   pass^4   pass^8
 0.50    0.500    0.938    0.996      0.500    0.062    0.004
 0.80    0.800    0.998    1.000      0.800    0.410    0.168
 0.90    0.900    1.000    1.000      0.900    0.656    0.430
 0.95    0.950    1.000    1.000      0.950    0.815    0.663
 0.99    0.990    1.000    1.000      0.990    0.961    0.923

5 attempts, 3 succeed: pass@1 0.60, pass@3 1.00, pass^3 0.10
```

## `traced_agent.py`

A calculator agent on Qwen2.5-1.5B-Instruct: 20 two-step word problems, 5 runs each at temperature 0.7, every model and tool call recorded as a span with OpenTelemetry's GenAI attribute names. Prints one line per task and saves the traces to `traces.json`. Its printed lines were not kept, so it has no `.out`.

In [ ]:
!{sys.executable} traced_agent.py

## `evaluate_traces.py`

Reads `traces.json` and the answer key: pass@k and pass^k scored two ways, each run's trajectory, calculator use, a dashboard of calls, tokens and latency, and one trace printed as a waterfall.

In [ ]:
!{sys.executable} evaluate_traces.py

The post printed:

```text
outcome scored as asked ('Answer: <number>'), 20 tasks x 5 runs:
  pass@1 0.15   pass^1 0.15
  pass@3 0.39   pass^3 0.00
  pass@5 0.55   pass^5 0.00
  tasks solved in all 5 runs: 0 of 20
outcome scored by the last number in the reply, 20 tasks x 5 runs:
  pass@1 0.93   pass^1 0.93
  pass@3 1.00   pass^3 0.81
  pass@5 1.00   pass^5 0.70
  tasks solved in all 5 runs: 14 of 20

trajectories (100 runs):
   81  right, the calculator produced the answer
   11  right, the calculator didn't produce it
    4  wrong: no calculator
    3  wrong: asked the calculator the wrong question
    1  cut off by the token budget (still right)
  calculator calls per run: none 14, one 84, two or more 2 (the instructions asked for one per arithmetic step: two)
  runs with no 'Answer:' line: 83; of those, replies saying 'the answer is': 19

dashboard: 100 runs, 186 model calls, 88 tool calls
  per run: 1.9 model calls, 454 tokens in, 81 out; per successful run: 575 tokens
  run latency p50 2.5 s, p95 10.9 s; model call p50 1.55 s; tool call p50 33 us

one trace (task 9, right answer 12975):
  invoke_agent warehouse-helper                0 ms  +  2468 ms  
      chat qwen2.5-1.5b-instruct                   1 ms  +  1594 ms  in 213 out 33 tokens, finish stop
      execute_tool calculator                   1595 ms  +     0 ms  {'expression': '8113-(423+7*755)'} -> 2405
      chat qwen2.5-1.5b-instruct                1596 ms  +   872 ms  in 268 out 12 tokens, finish stop
  final reply: 'The warehouse holds 2405 boxes now.'
```